# DINOv2-S/14 Linear Probe · 1,121장 복제 충돌 수정본

Drive의 `MyDrive/Cat_Project_Backup`에 `dinov2_duplicate_fix_source.zip`과 기존 `processed_1121.zip`을 둡니다. Colab에서 GPU 런타임을 선택하고 1 → 2 → 3 → 4번을 실행하세요.
저장된 모델 평가만 하려면 1 → 3번, 그래프만 보려면 1 → 4번을 실행합니다.
기존 실험 파일은 기본적으로 `results/dinov2-1121-01`에서 읽습니다. 거기에 E00–E23의 실제 설정 CSV나 JSON이 있어야 2번 셀이 학습을 진행합니다.

동일 원본이 서로 다른 샴고양이 개체 ID로 중복된 경우 격리합니다. 원본과 기존 결과를 보존합니다.


In [ ]:
# 1. Drive 연결, 수정된 소스 검증, 1,121장 복원 및 캐시 위치 확인
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
BACKUP = DRIVE / "Cat_Project_Backup"
SOURCE_ZIP = BACKUP / "dinov2_duplicate_fix_source.zip"
PROCESSED_ZIP = BACKUP / "processed_1121.zip"
DATA_ROOT = DRIVE  # 원본 세 클래스 폴더의 공통 상위 경로
LEGACY_RESULTS = BACKUP / "results" / "dinov2-1121-01"
RUN_ID = "dinov2-duplicate-fix-1121-01"
TRAIN_SEED = 42
TEST_SEED = None  # None이면 실행 시 난수 시드를 만들고 결과에 기록
N_PER_CLASS = 10
INPUT_SIZE = (224, 336, 448)
BATCH_SIZE = 32
EXPECTED = ("main_siamese_cat", "other_siamese_cat", "etc_cats")

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"Drive에 수정된 프로젝트 소스 ZIP을 올리세요. {SOURCE_ZIP}")
revision = sha256(SOURCE_ZIP.read_bytes()).hexdigest()
PROJECT = Path("/content") / f"dinov2_duplicate_fix_{revision[:12]}"
with ZipFile(SOURCE_ZIP) as package:
    release = json.loads(package.read("SOURCE_RELEASE.json"))
    if release["release"] != "dinov2-duplicate-fix-1121":
        raise ValueError("복제 충돌 수정본 소스 ZIP이 필요합니다.")
    names = package.namelist()
    if len(names) != len(set(names)) or set(names) != {*release["files"], "SOURCE_RELEASE.json"}:
        raise ValueError("소스 ZIP에 누락되거나 중복된 파일이 있습니다.")
    for name, expected_hash in release["files"].items():
        path = PurePosixPath(name)
        if path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name:
            raise ValueError("소스 ZIP에 안전하지 않은 경로가 있습니다.")
        data = package.read(name)
        if sha256(data).hexdigest() != expected_hash:
            raise ValueError(f"소스 SHA-256 불일치 · {name}")
    if not PROJECT.exists():
        with tempfile.TemporaryDirectory(dir=PROJECT.parent) as temporary:
            stage = Path(temporary) / "sources"
            package.extractall(stage)
            stage.rename(PROJECT)
for name, expected_hash in release["files"].items():
    if sha256((PROJECT / name).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"실행 폴더의 소스가 다운로드한 검토본과 다릅니다. {name}")

def python(*args, check=True, show_output=False):
    environment = {**os.environ, "TRAIN_SEED": str(TRAIN_SEED),
                   "BATCH_SIZE": str(BATCH_SIZE)}
    command = [sys.executable, "-u", *map(str, args)]
    if not show_output:
        return subprocess.run(command, cwd=PROJECT, env=environment, check=check)
    with subprocess.Popen(
        command, cwd=PROJECT, env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        result = subprocess.CompletedProcess(command, process.wait())
    if check:
        result.check_returncode()
    return result

SOURCE = next(
    (root for root in (DATA_ROOT, DATA_ROOT / "데이터")
     if all((root / name).is_dir() for name in EXPECTED)),
    None,
)
if SOURCE is None:
    raise FileNotFoundError("DATA_ROOT를 원본 세 클래스 폴더의 상위 경로로 설정하세요.")
PROCESSED = PROJECT / "outputs" / "processed_1121"
python("-m", "pip", "install", "-q", "-r", "requirements.txt")
python("-c", (
    "import torch; print('연산 장치', torch.cuda.get_device_name(0) "
    "if torch.cuda.is_available() else 'CPU')"
))
python("-m", "src.study_data", "--source", SOURCE, "--archive", PROCESSED_ZIP,
       "--output", PROCESSED)
print("검증된 소스 ZIP SHA-256", revision)
print("기존 임베딩 캐시 디렉터리 수", sum(p.is_dir() for p in (BACKUP / "embeddings").glob("*")))
print("결과 폴더", BACKUP / "results" / RUN_ID)


In [ ]:
# 2. 감사, E00–E23 재평가, 추가 탐색, 고정 평가와 최종 모델 저장
AUDIT_ONLY = False
ADDITIONAL_BUDGET = 31
THRESHOLD_POLICY = "max_f1"
FPR_TARGETS = (0.01, 0.05, 0.10)
SESSIONS_CSV = None  # 필요 시 확인한 촬영 목록 Path 입력
REVIEW_CSV = None  # 필요 시 확인한 유사 이미지 목록 Path 입력
RUN_UNIT_TESTS = False

python("-c", "import torch; assert torch.cuda.is_available(), 'GPU 런타임이 필요합니다.'")
if RUN_UNIT_TESTS:
    python("-m", "pytest", "-v")
args = [
    "-m", "src.generalization_study", "--source", SOURCE, "--processed", PROCESSED,
    "--output", BACKUP, "--run-id", RUN_ID, "--legacy-results", LEGACY_RESULTS,
    "--threshold-policy", THRESHOLD_POLICY, "--fpr-targets", *FPR_TARGETS,
    "--quarantine-other-copies",
    "--additional-budget", ADDITIONAL_BUDGET, "--input-sizes", *INPUT_SIZE,
]
if AUDIT_ONLY:
    args.append("--audit-only")
if SESSIONS_CSV is not None:
    args.extend(["--sessions", SESSIONS_CSV])
if REVIEW_CSV is not None:
    args.extend(["--review", REVIEW_CSV])
python(*args, show_output=True)


In [ ]:
# 3. 저장된 모델만 테스트
from IPython.display import Image, display

RUN_SINGLE = False
RUN_RANDOM = True
RUN_STRICT = True
RUN_RESERVED = True
RUN_REPRODUCIBILITY = False
IMAGE_PATH = None  # RUN_SINGLE=True일 때 원본 이미지 경로
EXTERNAL_SOURCE = None  # 필요 시 새 세 클래스 폴더의 상위 경로
STRICT_DEVELOPMENT = False
CHECKPOINT = BACKUP / "checkpoints" / RUN_ID / "final_model.pt"
TEST_RESULTS = BACKUP / "graphs" / RUN_ID / "sample_tests"

jobs = {
    "single": RUN_SINGLE, "random": RUN_RANDOM, "strict": RUN_STRICT,
    "reserved-all": RUN_RESERVED, "reproducibility": RUN_REPRODUCIBILITY,
    "external": EXTERNAL_SOURCE is not None,
}
failures = []
for mode, enabled in jobs.items():
    if not enabled:
        continue
    if mode == "reserved-all":
        import torch

        saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
        if not saved.get("holdout_indices"):
            print("고정 Holdout 미구성으로 전체 평가를 건너뜁니다.")
            continue
    args = [
        "-m", "src.test_study_model", "--mode", mode, "--source", SOURCE,
        "--processed", PROCESSED, "--output", BACKUP, "--run-id", RUN_ID,
        "--checkpoint", CHECKPOINT, "--n-per-class", N_PER_CLASS,
    ]
    if TEST_SEED is None and mode != "reproducibility":
        args.append("--random-seed")
    else:
        args.extend(["--test-seed", 123 if TEST_SEED is None else TEST_SEED])
    if IMAGE_PATH is not None:
        args.extend(["--image", IMAGE_PATH])
    if EXTERNAL_SOURCE is not None:
        args.extend(["--external-source", EXTERNAL_SOURCE])
    if STRICT_DEVELOPMENT:
        args.append("--strict-development")
    process = python(*args, check=False)
    if process.returncode:
        failures.append(mode)
        continue
    latest = max(TEST_RESULTS.glob(f"{mode}-*"), key=lambda path: path.stat().st_mtime_ns)
    print(latest / "results.json")
    if mode not in ("reserved-all", "external") and (latest / "predictions.png").is_file():
        display(Image(filename=str(latest / "predictions.png")))
if failures:
    print("완료되지 않은 테스트", failures, "· 위 로그에서 실패 이유를 확인하세요.")


In [ ]:
# 4. 저장된 표와 PNG 생성
import pandas as pd
from IPython.display import Image, display

python("-m", "src.plot_generalization", "--output", BACKUP, "--run-id", RUN_ID,
       "--processed", PROCESSED)
RESULTS = BACKUP / "results" / RUN_ID
GRAPHS = BACKUP / "graphs" / RUN_ID
for name in ("baseline_comparison.csv", "legacy_reevaluation.csv",
             "resolution_comparison.csv", "threshold_comparison.csv"):
    display(pd.read_csv(RESULTS / name))
for name in ("baseline_comparison.png", "fold_class_distribution.png",
             "overfitting_gap.png", "roc_curve.png", "confusion_matrix.png"):
    display(Image(filename=str(GRAPHS / name)))
print("CSV/JSON", RESULTS)
print("PNG", GRAPHS)
